In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
옵션을 사용하여 Estimator 프리미티브를 사용자 정의할 수 있습니다. 프리미티브의 `run()` 메서드 인터페이스는 모든 구현에서 공통적이지만 옵션은 그렇지 않습니다. [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) 및 [`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html) 옵션에 대한 정보는 API 참조를 참조하세요.

<span id="pass-options"></span>

## Estimator 옵션 설정

Estimator 초기화 시, 초기화 후, 또는 Estimator가 초기화된 후에 옵션을 업데이트할 수 있습니다. 이러한 기법을 사용하는 방법은 [옵션 소개](/guides/runtime-options-overview#options-precedence) 주제를 참조하세요.

또한 다음 섹션에서 설명하는 것처럼 `run()` 메서드에서 `precision` 값을 설정할 수 있습니다.
<span id="run-method"></span>

### run() 메서드

`run()`에 전달할 수 있는 값은 인터페이스에서 정의된 값들뿐입니다. 즉, Estimator의 경우 `precision`입니다. 이는 현재 실행에서 `default_precision`에 설정된 모든 값을 덮어씁니다.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### 특수 케이스: precision
`EstimatorV2.run` 메서드는 두 가지 인수를 받습니다: PUB 목록 (각각 precision에 대한 PUB별 값을 지정할 수 있음)과 precision 키워드 인수입니다. 이 precision 값들은 Estimator 실행 인터페이스의 일부이며 Runtime Estimator의 옵션과 독립적입니다. Estimator 추상화를 준수하기 위해 옵션으로 지정된 모든 값보다 우선합니다.

그러나 PUB 또는 run 키워드 인수에서 `precision`이 지정되지 않은 경우 (또는 모두 `None`인 경우), 옵션의 precision 값이 사용되며, 특히 `default_precision`이 사용됩니다.

Estimator 옵션에는 `default_shots`와 `default_precision`이 모두 포함되어 있습니다. 그러나 gate-twirling이 기본적으로 활성화되어 있으므로 `num_randomizations`와 `shots_per_randomization`의 곱이 이 두 옵션보다 우선합니다.

구체적으로 Estimator PUB에 대해:

1. PUB가 precision을 지정한 경우, 그 값을 사용합니다.
2. `run`에서 precision 키워드 인수가 지정된 경우, 그 값을 사용합니다.
3. `twirling`이 활성화된 경우 (기본적으로 True), [`twirling` 옵션](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)에서 지정된 `num_randomizations`와 `shots_per_randomization`의 곱이 사용됩니다.
4. `estimator.options.default_shots`가 지정된 경우, 그 값을 사용하여 데이터 양을 제어합니다.
5. `estimator.options.default_precision`이 지정된 경우, 그 값을 사용합니다.

예를 들어, 네 곳 모두에서 precision이 지정된 경우 가장 높은 우선순위를 가진 것 (PUB에서 지정된 precision)이 사용됩니다.

> **Note:** Precision은 사용량과 반비례합니다. 즉, precision이 낮을수록 실행에 더 많은 QPU 시간이 걸립니다.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## 모든 오류 완화 및 오류 억제 끄기
예를 들어 자체적인 완화 기법을 연구하고 있는 경우 모든 오류 완화 및 억제를 끌 수 있습니다. 이를 위해 `resilience_level = 0`을 설정하세요.

예시:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## 사용 가능한 옵션
다음 표는 `qiskit-ibm-runtime`의 최신 버전에서 사용 가능한 옵션을 문서화합니다. 이전 옵션 버전을 보려면 [`qiskit-ibm-runtime` API 참조](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime)를 방문하여 이전 버전을 선택하세요.

<Accordion>
<AccordionItem title="`default_shots`">

circuit 구성당 사용할 총 shot 수.

**선택지**: 정수 >= 0

**기본값**: None

[`default_shots` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

precision을 지정하지 않는 PUB 또는 `run()` 호출에 사용할 기본 precision.

**선택지**: 부동 소수점 > 0

**기본값**: 0.015625 (1 / sqrt(4096))

[`default_precision` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

dynamical decoupling 오류 완화 설정을 제어합니다.

[`dynamical_decoupling` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**선택지**: `True`, `False`

**기본값**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**선택지**: `middle`, `edges`

**기본값**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

선택지: `asap`, `alap`
기본값: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

선택지: `XX`, `XpXm`, `XY4`
기본값: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

선택지: `True`, `False`
기본값: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[`environment` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

`Job ID`와 `Job result`를 받는 호출 가능 함수.

**선택지**: None

**기본값**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

태그 목록.

**선택지**: None

**기본값**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**선택지**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**기본값**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**선택지**: `True`, `False`

**기본값**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[`execution` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
각 shot에 대해 qubit을 기저 상태로 초기화할지 여부.

**선택지**: `True`, `False`

**기본값**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
측정과 이후 양자 circuit 사이의 지연.

**선택지**: `backend.rep_delay_range`에서 제공하는 범위의 값

**기본값**: `backend.default_rep_delay`로 지정
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
작업 실행 시간 제한 (초). 자세한 내용은 [최대 실행 시간](/guides/max-execution-time) 가이드를 참조하세요.

**선택지**: [1, 10800] 범위의 정수 (초)

**기본값**: 10800 (3시간)
  </AccordionItem>

<AccordionItem title="`resilience`">
resilience 전략을 미세 조정하기 위한 고급 resilience 옵션.

[`resilience` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

레이어 노이즈 학습 옵션.

[`resilience.layer_noise_learning` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**선택지**: [0, 200] 범위의 2-10개 값의 list[int]

**기본값**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**선택지**: None, 정수 >= 1

**기본값**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**선택지**: 정수 >= 1

**기본값**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**선택지**: 정수 >= 1

**기본값**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**선택지**: `NoiseLearnerResult`, `Sequence[LayerError]`

**기본값**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**선택지**: `True`, `False`

**기본값**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

측정 노이즈 학습 옵션.

[`resilience.measure_noise_learning` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**선택지**: 정수 >= 1

**기본값**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**선택지**: 정수, `auto`

**기본값**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**선택지**: `True`, `False`

**기본값**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

확률적 오류 취소 완화 옵션.

[`resilience.pec` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**선택지**: `None`, 정수 >= 1

**기본값**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**선택지**: `auto`, [0, 1] 범위의 부동 소수점

**기본값**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**선택지**: `True`, `False`

**기본값**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[`resilience.zne` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**선택지**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**기본값**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**선택지**: 부동 소수점 목록

**기본값**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**선택지**: 다음 중 하나 이상: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**기본값**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**선택지**: 부동 소수점 목록; 각 부동 소수점 >= 1

**기본값**: `PEA`의 경우 `(1, 1.5, 2)`, 그 외에는 `(1, 3, 5)`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

오류에 대한 resilience 수준. 수준이 높을수록 더 긴 처리 시간을 대가로 더 정확한 결과를 얻습니다. 자세한 내용은 Noise management 주제의 [resilience 수준](/guides/estimator-noise-management#resilience) 섹션을 참조하세요.

**선택지**: `0`, `1`, `2`

**기본값**: `1`

[`resilience_level` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**선택지**: 정수

**기본값**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

백엔드를 시뮬레이션할 때 전달할 옵션

[`simulator` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**선택지**: 풀어낼 기저 게이트 이름 목록

**기본값**: [Qiskit Aer 시뮬레이터](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)가 지원하는 모든 기저 게이트 세트

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**선택지**: 방향성 있는 2-qubit 상호작용 목록

**기본값**: None (연결 제약 없음, 완전 연결을 의미)

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**선택지**: [Qiskit Aer NoiseModel](/guides/build-noise-models), 또는 그 표현

**기본값**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**선택지**: 정수

**기본값**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

Twirling 옵션

[`twirling` API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**선택지**: True, False

**기본값**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**선택지**: True, False

**기본값**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**선택지**: `auto`, 정수 >= 1

**기본값**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**선택지**: `auto`, 정수 >= 1

**기본값**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**선택지**: `active`, `active-circuit`, `active-accum`, `all`

**기본값**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

실험적 옵션 (사용 가능한 경우).

  </AccordionItem>

</Accordion>
## 기능 호환성
특정 런타임 기능은 단일 작업에서 함께 사용할 수 없습니다. 선택한 기능과 호환되지 않는 기능 목록을 보려면 적절한 탭을 클릭하세요:

<Tabs>

  <TabItem value="Fractional" label="분수 게이트">
  다음과 호환되지 않습니다:
  - 게이트 twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="게이트 폴딩 ZNE">
    다음과 호환되지 않습니다:
  - PEA
  - PEC

  커스텀 게이트를 사용할 때 작동하지 않을 수 있습니다.
  </TabItem>
  <TabItem value="Twirling" label="게이트 twirling">
  분수 게이트 또는 스트레치와 호환되지 않습니다.

  기타 참고 사항:
  - 측정 twirling은 터미널 측정에만 적용할 수 있습니다.
  - 비-Clifford 엔탱글러와 작동하지 않습니다.

  </TabItem>

  <TabItem value="PEA" label="PEA">
    다음과 호환되지 않습니다:
  - 분수 게이트
  - 게이트 폴딩 ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    다음과 호환되지 않습니다:
  - 분수 게이트
  - 게이트 폴딩 ZNE
  - PEA
  </TabItem>

</Tabs>
## 다음 단계
> **Tip:** - [옵션 소개](/guides/runtime-options-overview) 가이드를 검토합니다.
> - `EstimatorV2` 메서드에 대한 자세한 내용은 [Estimator API 참조](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2)를 참조하세요.
> - 작업을 실행할 [실행 모드](/guides/execution-modes)를 결정합니다.
> - [Estimator를 사용한 노이즈 관리](/guides/estimator-noise-management)에 대해 알아봅니다.

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>